# Exercise 5: Model Versioning & Packaging

In this exercise, we will:
1. Register our tuned model in MLflow Model Registry
2. Merge LoRA weights with the base model for efficient serving
3. Create a Docker container for serving the model with vLLM
4. Build and test the container locally

---

## Prerequisites

Before starting this exercise, make sure you have completed:
- Exercise 1: Setup & Exploration
- Exercise 2: Data Preparation
- Exercise 3: LoRA Tuning
- Exercise 4: Evaluation

You should have a trained LoRA adapter saved and tracked in MLflow.

## 1. Environment Setup

Let's first check our environment and import necessary libraries.

In [ ]:
# Import necessary libraries
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import mlflow
import mlflow.pytorch
from pathlib import Path
from getpass import getpass

# Get Hugging Face token
hf_token = os.getenv("HF_TOKEN", None)
if hf_token:
    print("HF Token found")
else:
    print("Warning: No HF Token provided. You may encounter rate limits.")

# Check if we have GPU available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.current_device()}")
else:
    print("Current device: CPU")


## 2. Load the Trained LoRA Model from Local Disk

In this section, we will load our trained LoRA adapter from local disk, where it was saved during the training exercise (notebook 03). We will then log it to MLflow for versioning.

In [ ]:
# Set up MLflow tracking
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Experiment name used during training
experiment_name = "llm-lora-tuning"

# Get or create the experiment
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"Created new experiment: {experiment_name} (ID: {experiment_id})")
else:
    experiment_id = experiment.experiment_id
    print(f"Experiment ID: {experiment_id}")

# Load the trained LoRA adapter from local disk (where notebook 03 saved it)
adapter_path = "./lora_adapter"
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading base model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

tokenizer = AutoTokenizer.from_pretrained(
    base_model_name,
    token=hf_token,
)

print("Loading LoRA adapter from local disk...")
model = PeftModel.from_pretrained(base_model, adapter_path)
print("Model loaded successfully!")
print(f"Model type: {type(model).__name__}")

## 3. Register Model in MLflow Model Registry

Now we'll register our trained model in the MLflow Model Registry for versioning and lifecycle management.

In [ ]:
# Register the model in MLflow Model Registry
model_name = "tinyllama-lora-bike-assistant"
client = mlflow.tracking.MlflowClient()

# Start a new MLflow run to log the model
with mlflow.start_run(experiment_id=experiment_id, run_name="model-registration") as run:
    run_id = run.info.run_id
    print(f"MLflow Run ID: {run_id}")

    # Log the LoRA adapter directory as an artifact
    mlflow.log_artifact(adapter_path, artifact_path="lora_adapter")

    # Log model parameters
    mlflow.log_param("base_model", base_model_name)
    mlflow.log_param("adapter_path", adapter_path)

    # Log the PEFT model to MLflow
    mlflow.pytorch.log_model(
        model,
        artifact_path="model",
        pip_requirements=["torch", "transformers", "peft", "accelerate", "bitsandbytes"]
    )
    print("Model weights logged to MLflow artifacts")

    # Create or get the registered model
    try:
        client.create_registered_model(model_name)
        print(f"Created new registered model: {model_name}")
    except Exception:
        print(f"Registered model {model_name} already exists")

    # Create a new model version pointing to the logged model
    model_uri = f"runs:/{run_id}/model"
    model_version = client.create_model_version(
        name=model_name,
        source=model_uri,
        run_id=run_id
    )
    print(f"Model version {model_version.version} created for {model_name}")

    # Transition the model to Staging
    client.transition_model_version_stage(
        name=model_name,
        version=model_version.version,
        stage="Staging",
        archive_existing_versions=False
    )
    print(f"Model version {model_version.version} transitioned to Staging")

## 4. Merge LoRA Weights with Base Model

For serving efficiency, we merge the LoRA adapter weights with the base model. This creates a single model file that doesn't require the PEFT library at inference time.

In [ ]:
# Merge LoRA weights with base model for efficient serving
print("Merging LoRA weights with base model...")

# Merge the LoRA layers into the base model
merged_model = model.merge_and_unload()

# Save the merged model to the location expected by the Dockerfile
merged_model_path = "../models/merged_model"
os.makedirs(merged_model_path, exist_ok=True)

merged_model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)

print(f"Merged model saved to: {merged_model_path}")

# Verify the merged model loads correctly
print("Verifying merged model...")
test_model = AutoModelForCausalLM.from_pretrained(
    merged_model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
test_tokenizer = AutoTokenizer.from_pretrained(
    merged_model_path,
)
print("Merged model verified successfully!")

## 5. Create Build and Push Script

A helper script `build_and_push.sh` for building and pushing the Docker image to a container registry exists in the `../scripts` directory .

## 6. Create MLflow Registration Script

Now we'll create a reusable CLI script for registering models in MLflow. This script can be run from the terminal to register future models without using the notebook.

The `mlflow_register.py` script already exists in the `../scripts/` directory.

## 7. Verify the Scripts

Let's verify that the scripts were created correctly.

In [ ]:
import os

# Check if scripts exist
scripts_dir = os.path.join('..', 'scripts')
if os.path.exists(scripts_dir):
    files = os.listdir(scripts_dir)
    print(f"Scripts in {scripts_dir}:")
    for f in files:
        path = os.path.join(scripts_dir, f)
        size = os.path.getsize(path)
        print(f"  - {f} ({size} bytes)")
else:
    print(f"Directory {scripts_dir} does not exist yet.")

## 8. Clean Up Temporary Files

Let's clean up the temporary tokenizer directory we created.

In [ ]:
import shutil

# Remove temporary tokenizer directory
if os.path.exists('./tokenizer'):
    shutil.rmtree('./tokenizer')
    print("Cleaned up temporary tokenizer directory")

---

## Summary

In this exercise, we learned how to:
1. Register a fine-tuned model in MLflow Model Registry
2. Merge LoRA adapter weights with the base model for efficient serving
3. Create a build and push script for containerization
4. Create a reusable MLflow registration script

## Key Takeaways

- **MLflow Model Registry** provides version control and lifecycle management for ML models
- **LoRA weight merging** eliminates the PEFT dependency at inference time, reducing complexity
- **Containerization** packages the model with all dependencies for reproducible deployment
- **Script automation** ensures consistent and repeatable model registration and packaging

---

**Next: Proceed to Exercise 6 - Deployment & Serving**